# 02 — Structural Descriptor State → Compression Curve / Performance

첨부 `Com_Training_Visualization_v22`의 핵심 철학을 반영합니다: curve parser, 공통 strain grid, event-aware densification/plateau/energy property extraction, design-group-safe CV, curve latent + direct property heads, uncertainty.

**권장 입력:** long format `design_id, specimen_id, replicate_id, strain, stress_MPa`. 기존 v22의 5-column Excel block 폴더도 adapter로 읽을 수 있습니다.


In [ ]:
from pathlib import Path
import sys,os,json
PROJECT_POINTER=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation\.ai_voxel_ml_project.json")
CODE_DIR=Path.cwd()/"Code" if (Path.cwd()/"Code").exists() else Path.cwd();sys.path.insert(0,str(CODE_DIR)) if str(CODE_DIR) not in sys.path else None
CURVE_INPUT_MODE='canonical_long' # canonical_long | legacy_v22_folder
COMPRESSION_LONG_FILE=None # None -> persistent project compression_curves_master.csv
LEGACY_V22_EXCEL_FOLDER=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Compression test")
GRID_POINTS=301;GRID_MAX=None;PREFER_FINAL_DESCRIPTOR=True;CV_SPLITS=5;RANDOM_SEED=42;N_JOBS=-1;RESUME=True


In [ ]:
import numpy as np,pandas as pd,joblib
from voxel_ml_common import load_contract,load_all_structures,mark_stage
from compression_curve_model import load_canonical_long,load_legacy_v22_excel_folder,build_curve_dataset
c=load_contract(PROJECT_POINTER);model_root=Path(c['model_root']);out=model_root/'02_desc2curve';out.mkdir(parents=True,exist_ok=True)
gen=joblib.load(model_root/'01_gen2desc'/'gen2desc_bundle.joblib');all_df=load_all_structures(c)
curve_file=Path(c['compression_curve_master']) if COMPRESSION_LONG_FILE is None else Path(COMPRESSION_LONG_FILE)
curves=load_canonical_long(curve_file) if CURVE_INPUT_MODE=='canonical_long' else load_legacy_v22_excel_folder(LEGACY_V22_EXCEL_FOLDER)
table,Y,M,grid=build_curve_dataset(curves,all_df,gen,GRID_POINTS,GRID_MAX,PREFER_FINAL_DESCRIPTOR)
table.to_csv(out/'curve_dataset_table.csv',index=False,encoding='utf-8-sig');np.savez_compressed(out/'curve_dataset_arrays.npz',Y=Y,M=M,grid=grid)
mark_stage(out,'01_curve_dataset','completed',[out/'curve_dataset_table.csv',out/'curve_dataset_arrays.npz'],{'curves':len(table),'designs':table.design_id.nunique()})
print('Matched curves:',len(table),'| unique designs:',table.design_id.nunique(),'| grid:',len(grid));display(table.head())


In [ ]:
import numpy as np,pandas as pd,joblib
from voxel_ml_common import load_contract,mark_stage
from compression_curve_model import train_curve_model
c=load_contract(PROJECT_POINTER);out=Path(c['model_root'])/'02_desc2curve';bundle_path=out/'desc2curve_bundle.joblib';table=pd.read_csv(out/'curve_dataset_table.csv');a=np.load(out/'curve_dataset_arrays.npz');Y=a['Y'];M=a['M'];grid=a['grid']
if RESUME and bundle_path.exists():bundle=joblib.load(bundle_path);cm=pd.read_csv(out/'curve_cv_metrics.csv');pm=pd.read_csv(out/'property_cv_metrics.csv')
else:bundle,cm,pm=train_curve_model(table,Y,M,grid,out,CV_SPLITS,RANDOM_SEED,N_JOBS)
mark_stage(out,'02_train_curve_surrogate','completed',[bundle_path,out/'curve_cv_metrics.csv',out/'property_cv_metrics.csv'],{'best_curve':bundle['best_curve_model'],'best_property':bundle['best_property_model']})
display(cm);display(pm)


In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,joblib
from voxel_ml_common import load_contract
c=load_contract(PROJECT_POINTER);out=Path(c['model_root'])/'02_desc2curve';o=np.load(out/'oof_curve_best.npz',allow_pickle=True);Y=o['Y_true'];P=o['Y_pred'];grid=o['grid'];ids=o['specimen_id']
fig,ax=plt.subplots(figsize=(8,5))
show=np.linspace(0,len(Y)-1,min(8,len(Y)),dtype=int)
for i in show:
    ax.plot(grid,Y[i],lw=1.4,alpha=.7,label=f'{ids[i]} true')
    ax.plot(grid,P[i],'--',lw=1.1,alpha=.8)
ax.set_xlabel('Compressive strain');ax.set_ylabel('Stress (MPa)');ax.set_title('Group-CV OOF curves: solid=true, dashed=prediction');ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(out/'oof_curve_examples.png',dpi=220);plt.show()
pp=pd.read_csv(out/'oof_property_best.csv');cols=[c[:-6] for c in pp.columns if c.endswith('__true')]
if cols:
    fig,axes=plt.subplots(1,min(3,len(cols)),figsize=(5*min(3,len(cols)),4));axes=np.atleast_1d(axes)
    for ax,c0 in zip(axes,cols[:3]):
        a=pp[c0+'__true'];b=pp[c0+'__pred'];ax.scatter(a,b,s=30);lo=min(a.min(),b.min());hi=max(a.max(),b.max());ax.plot([lo,hi],[lo,hi],'--');ax.set_title(c0.replace('perf__',''));ax.set_xlabel('Measured');ax.set_ylabel('OOF predicted');ax.grid(alpha=.2)
    fig.tight_layout();fig.savefig(out/'oof_property_parity.png',dpi=220);plt.show()
print('Saved OOF validation figures to',out)


### v22와의 연결
- `modulus`, `yield_stress`, `compressive/peak stress`, `plateau_stress`, `densification_strain`, `absorbed_energy_to_densification`, `cfe`를 curve에서 추출합니다.
- full curve는 PCA latent로 학습하고, scalar properties는 별도 direct head로 학습합니다.
- 이후 inverse optimizer는 **curve target과 property target을 동시에** 사용할 수 있습니다.
